# OpenCV project
## creation of a simple paint application with
## operations {**p = pen, e = eraser, d = delete ,  s = save and q = quit.**}
## Shapes of **Rectangle, Circle and line** on the right of the canva

In [ ]:
## my Opencv project

import cv2
import numpy as np

## creation of a white background
white = np.full((600,1200,3),255,dtype = np.uint8)
cv2.namedWindow("canva",cv2.WINDOW_FULLSCREEN) ## creating a namedWindow "canva"


def f1(x): ## to make Trakbar alive
    pass

# trackbars to adjust the intensities of the color.
cv2.createTrackbar("blue", "canva", 0, 255,f1)
cv2.createTrackbar("green", "canva",0, 255,f1)
cv2.createTrackbar("red", "canva", 0, 255,f1)
cv2.createTrackbar("thickness", "canva", 1, 100, f1)

def trac(): ## function to get the trackbars values.
    bl = cv2.getTrackbarPos("blue", "canva")
    gr = cv2.getTrackbarPos("green", "canva")
    re = cv2.getTrackbarPos("red", "canva")
    th = cv2.getTrackbarPos("thickness", "canva")
    return [(bl,gr,re), th]


x1,y1 = None, None
drawing = False
mode = "Pen"
shape_preview = None

# shapes positions on sidebar (right side.)
shapes = {
    "Rectangle": (1070, 70, 1130, 110),   # rectangle outline
    "Circle": (1100, 160, 30),            # circle center + radius
    "Line": ((1070, 220), (1130, 260))    # line endpoints
}


canva_bound = 1050 # to ensure that drawing area don't include the shapes area.

## function to track the mouse actions.
def f2(event,x,y,flags,user):
    global x1, y1, drawing, mode, shape_preview, canva_bound

    if event == 1: ## tracks mouse left click

        ## --- to check whether click is inside the shapes area.--- 
        Rx1, Ry1, Rx2, Ry2 = shapes["Rectangle"]
        Cx, Cy, r = shapes["Circle"]
        (Lx1,Ly1), (Lx2,Ly2) = shapes["Line"]
        
        ## --- assigns the mode value corresponding to the click position.---
        if Rx1 <= x <= Rx2 and Ry1 <= y <= Ry2:
            mode = "Rectangle"
        elif ((x-Cx)**2)+((y-Cy)**2) <= r**2:
            mode = "Circle"
        elif Lx1 <= x <= Lx2 and Ly1 <= y <= Ly2:
            mode = "Line"

        ## --- if click is inside the drawing area.
        elif x <= canva_bound:
            drawing = True ## if click inside the canva then only drawing is activated.
            x1 = x
            y1 = y
            shape_preview = white.copy() ## to track the drawing preview.

    elif event == 0 and drawing == True: ## if the mouse move after the click on drawing area.
        shape_preview = white.copy()
        color, thick = trac()
        
        if mode == "Pen":
            cv2.line(white, (x1,y1), (x,y), color, thick) ## directly on the white background
            x1, y1 = x, y
            
        elif mode == "Eraser":
            cv2.line(white, (x1,y1), (x,y), (255,255,255), thick)  ## directly on the white background
            x1, y1 = x, y
            
        elif mode == "Rectangle":
            cv2.rectangle(shape_preview, (x1,y1), (x,y), color, thick) ## on preview to display the movement.(temporay)
            
        elif mode == "Circle":
            radius = int(((x - x1)**2 + (y - y1)**2) ** 0.5)
            cv2.circle(shape_preview, (x1,y1), radius, color, thick) ## on preview to display the movement.(temporay)

        elif mode == "Line":
            cv2.line(shape_preview, (x1,y1), (x,y), color, thick) ## on preview to display the movement.(temporay)
            

    elif event == 4 and drawing == True: ## when the left click is released
        color, thick = trac()
        
        if mode == "Rectangle":
            cv2.rectangle(white, (x1,y1), (x,y), color, thick) ## making the rectangle permanet in white canva
            
        elif mode == "Circle":
            radius = int(((x - x1)**2 + (y-y1)**2)**0.5)
            cv2.circle(white, (x1,y1), radius, color, thick) ## making the circle permanet in white canva

        elif mode == "Line":
            cv2.line(white, (x1,y1), (x,y), color, thick) ## making the circle permanet in white canva
            
        drawing = False    
        x1, y1 = None, None
        shape_preview = None
        
    
cv2.setMouseCallback("canva",f2) ## to call the f2 and track the mouse actions.

i = 1
while True:
    display = white.copy() if shape_preview is None else shape_preview.copy()

    ## displaying the options to the user.
    cv2.putText(display, "MODES = {Q : Quit | P : Pen | E : eraser | S : Save | D : Delete}", (10,540),
                cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0,0,0),2)
    ## displayinf the selected option to the user.
    cv2.putText(display, f"MODE = {mode}", (10,580), cv2.FONT_HERSHEY_PLAIN, 1, (0,150,0),2)

    ## displaying the shapes on the canva
    Rx1,Ry1,Rx2,Ry2 = shapes["Rectangle"]
    cv2.putText(display, "Shapes", (Rx1,Ry1 - 20), cv2.FONT_HERSHEY_PLAIN, 1, (0,0,0),2)
    cv2.rectangle(display, (Rx1,Ry1), (Rx2,Ry2), (0,0,0), 2)
    Cx1,Cx2,r = shapes["Circle"]
    cv2.circle(display, (Cx1,Cx2), r, (0,0,0), 2)
    (Lx1,Ly1),(Lx2,Ly2) = shapes["Line"]
    cv2.line(display, (Lx1,Ly1), (Lx2,Ly2), (0,0,0), 2)

    ## linking the namedWindow canva to the display 
    cv2.imshow("canva",display)
    key = cv2.waitKey(1)

    ## Modes to perform actions.
    if key in [ord("q"), ord("Q")]:
        break
    elif key in [ord("p"), ord("P")]:
        mode = "Pen"
    elif key in [ord("e"), ord("E")]:
        mode = "Eraser"
    elif key in [ord("S"), ord("s")]:
        path = '/Users/vidyasagar/Desktop/DataScience/INNOMATICS/VISION/Jupyter/project/screenshot_{}.jpg'.format(i)
        cv2.imwrite(path,white)
        i+=1
        mode = f"saved :{path}"
        print(mode)
    elif key in [ord("D"), ord("d")]:
        ## to delete everything on the display : assigning all the array values with 255 to make it white again.
        white[:] = 255
        mode = "Deleted"

cv2.destroyAllWindows()

saved :/Users/vidyasagar/Desktop/DataScience/INNOMATICS/VISION/Jupyter/project/screenshot_1.jpg
